In [1]:
!wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

--2026-04-05 09:43:13--  https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/87156914/0b363e00-0126-11e9-9e3c-e8c235463bd6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-05T10%3A43%3A42Z&rscd=attachment%3B+filename%3Dlenta-ru-news.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-05T09%3A43%3A13Z&ske=2026-04-05T10%3A43%3A42Z&sks=b&skv=2018-11-09&sig=gH%2BZ33X3TR%2BFRtLZ0XmZwOOplLPsNrbF56xMklnRYgk%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NTM4NTc5NCwibmJmIjoxNzc1MzgyMTk0LCJwYXRoIjoicmVsZWFzZWFzc2

In [2]:
!pip install corus
!python3 -m spacy download ru_core_news_sm
!pip install bertopic hdbscan sentence-transformers
!pip install razdel
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 88.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 79.8 MB/s eta 0:00:00


In [3]:
from corus import load_lenta
import spacy
import string
from tqdm import tqdm
import string
import pandas as pd
from razdel import tokenize, sentenize
import pymorphy3
import numpy as np
import re
import random

import nltk

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve
    )
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer

from functools import lru_cache
from sklearn.base import clone

from tempfile import mkdtemp
from joblib import Memory

import urllib.request

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.dimensionality import BaseDimensionalityReduction

from time import perf_counter

from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer

Получим массив в 10.000 текстов из датасета лента ру

In [4]:
from typing_extensions import dataclass_transform
path = 'lenta-ru-news.csv.gz'
records = load_lenta(path)
data_sup = []

for i, record in enumerate(tqdm(records, total=100000)):
    if i >= 100000:
        break
    data_sup.append({
        'text': record.text,
        'title': record.title,
        'topic': record.topic
    })

data = pd.DataFrame(data_sup)

print(data.info())
data.head()

100%|██████████| 100000/100000 [00:04<00:00, 23595.79it/s]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    100000 non-null  object
 1   title   100000 non-null  object
 2   topic   100000 non-null  object
dtypes: object(3)
memory usage: 2.3+ MB
None


,text,title,topic
0,Вице-премьер по социальным вопросам Татьяна Го...,Названы регионы России с самой высокой смертно...,Россия
1,Австрийские правоохранительные органы не предс...,Австрия не представила доказательств вины росс...,Спорт
2,Сотрудники социальной сети Instagram проанализ...,Обнаружено самое счастливое место на планете,Путешествия
3,С начала расследования российского вмешательст...,В США раскрыли сумму расходов на расследование...,Мир
4,Хакерская группировка Anonymous опубликовала н...,Хакеры рассказали о планах Великобритании зами...,Мир


Определим количество топиков на которые размечен сам датасет, что бы ориетироваться также на эти данные при оценке кластеризации

In [5]:
data['topic'].value_counts().__len__()

19

In [6]:
data = data['text']

In [7]:
train_docs, holdout_docs = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

In [8]:
train_docs = train_docs.to_list()

In [9]:
embedding_model = SentenceTransformer('all-MiniLM-L12-v2', device='cuda')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

При поппытке поставить классические значения n_neighbors в диапазоне 10-20 дают большое количество создаваемых кластеров, что сильно привосходит количество в оригинальной рзаметке датасета по топикам. Если пробовать получить схожее количество тем, то стоит взять количество соседий и компонент большим при понижении размерности

In [10]:
dim_model = UMAP(n_neighbors=50, n_components=20, min_dist=0.0, metric='cosine')

In [12]:
cluster_model = HDBSCAN(min_cluster_size=35, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

In [11]:
nltk.download('stopwords')
from nltk.corpus import stopwords

russian_stop_words = stopwords.words('russian')

vectorizer_model = CountVectorizer(stop_words=russian_stop_words, min_df=5)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [12]:
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [13]:
representation_model = KeyBERTInspired()

In [16]:
topic_model = BERTopic(
  embedding_model=embedding_model,
  umap_model=dim_model,
  hdbscan_model=cluster_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model,
  representation_model=representation_model,
  calculate_probabilities=True,
)

In [17]:
%%time

topics, probs = topic_model.fit_transform(train_docs)

CPU times: user 15min 30s, sys: 4.79 s, total: 15min 34s
Wall time: 12min 6s


In [18]:
topic_info = topic_model.get_topic_info()
num_topics = len(topic_info) - 1
print(f"Количество тем (без учета шума): {num_topics}")

Количество тем (без учета шума): 139


In [19]:
topic_model.visualize_barchart(top_n_topics=40, n_words=10, title='Топ слов по темам')

Замерим количество сэмплов отмеченных, как шум. Далее постараемся повысить покрытия моделью тематического моделирования исходного сета данных

In [20]:
def coverage_stats(topic_list):
    arr = np.asarray(topic_list)
    total = len(arr)
    assigned = int(np.sum(arr != -1))
    outliers = int(np.sum(arr == -1))
    return {
        "total_docs": total,
        "assigned_docs": assigned,
        "outliers": outliers,
        "coverage": assigned / total,
        "outlier_rate": outliers / total
    }

before_coverage = pd.DataFrame([coverage_stats(topics)], index=["before"])
before_coverage

,total_docs,assigned_docs,outliers,coverage,outlier_rate
before,80000,52707,27293,0.658837,0.341162


In [21]:
reduced_topics_ctfidf = topic_model.reduce_outliers(
    train_docs,
    topics,
    strategy="c-tf-idf",
    threshold=0.1
)

after_coverage_ctfidf = pd.DataFrame([coverage_stats(reduced_topics_ctfidf)], index=["after_c_tf_idf"])
pd.concat([before_coverage, after_coverage_ctfidf])

,total_docs,assigned_docs,outliers,coverage,outlier_rate
before,80000,52707,27293,0.658837,0.341162
after_c_tf_idf,80000,59920,20080,0.749000,0.251000


In [25]:
topic_distr, topic_token_distr = topic_model.approximate_distribution(train_docs[:10000], calculate_tokens=True, batch_size=100)

In [27]:
df = topic_model.visualize_approximate_distribution(train_docs[301], topic_token_distr[301])
df

,Президент,США,Дональд,Трамп,своем,Twitter,дал,обещание,опубликовать,все,документы,по,делу,об,убийстве,35,го,лидера,США,Джона,Кеннеди,После,серьезных,консультаций,генералом,Келли,ЦРУ,другими,агентствами,опубликую,все,документы,по,Джону,Кеннеди,за,исключением,тех,которых,упоминаются,имена,адреса,еще,живых,людей,написал,Трамп,Президент,США,отметил,что,сделает,это,целью,положить,конец,всем,теориям,заговора,26,октября,Трамп,подписал,меморандум,публикации,почти,трех,тысяч,неизвестных,документов,об,убийстве,Джона,Кеннеди,При,этом,он,подчеркнул,что,будут,обнародованы,не,все,документы,объясняя,это,просьбой,федеральных,агентств,ведомств,дать,им,время,до,26,апреля,2018,года,на,дополнительное,ознакомление,материалами,Джон,Кеннеди,руководил,США,чуть,более,двух,лет,был,смертельно,ранен,во,время,поездки,Даллас,22,ноября,1963,года,Согласно,официальной,версии,его,убийцей,был,одиночка,по,имени,Ли,Харви,Освальд,стрелявший,президента,из,снайперской,винтовки,большого,расстояния,Обстоятельства,смерти,президента,до,сих,пор,вызывают,споры
113_палестинцы_позитивный_конструктивный_президенты,0.109,0.109,0.109,0.109,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


In [28]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def topic_diversity(topic_words, top_n=10):
    all_words = []
    for words in topic_words:
        all_words.extend(words[:top_n])

    unique_words = len(set(all_words))
    total_words = len(all_words)

    return unique_words / total_words if total_words > 0 else 0


topic_words = []
for topic_id in range(len(topic_model.get_topic_info()) - 1):  # -1 чтобы исключить -1 (шум)
    words = topic_model.get_topic(topic_id)
    if words:
        top_words = [word for word, _ in words[:10]]  # берем топ-10 слов
        topic_words.append(top_words)

diversity_score = topic_diversity(topic_words, top_n=10)
print(f"Topic Diversity: {diversity_score:.3f}")

Topic Diversity: 0.819


In [30]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def umass_coherence_gensim(topic_words, texts, dictionary=None, top_n=10):
    if dictionary is None:
        dictionary = Dictionary(texts)

    coherence_model = CoherenceModel(
        topics=[[word for word, _ in topic[:top_n]] for topic in topic_words],
        texts=texts,
        dictionary=dictionary,
        coherence='u_mass'
    )

    return coherence_model.get_coherence()



tokenized_texts = [doc.split() for doc in train_docs]

bertopic_topics = []
for topic_id in range(len(topic_model.get_topic_info()) - 1):
    words = topic_model.get_topic(topic_id)
    if words:
        bertopic_topics.append(words)

# Расчет UMass Coherence
umass_score = umass_coherence_gensim(
    bertopic_topics,
    tokenized_texts,
    top_n=10
)
print(f"UMass Coherence: {umass_score:.3f}")

UMass Coherence: -9.745


метрика topic_diversity принимает неплохие значения, однако когерентность топиков = -9.745 означает, что класстеры состоят из хаотичных наборов слов и тем, которые не имеют между собой четкого различия

Для снижения количества топиков и снижения между ними когерентности можно попробовать поставить более агресивные параметры класстеризации. Увеличить минимальный размер кластера, а также установить нижний трешхолд по количеству сэмплов в одном кластере. Сделаем это и переобучим модель тематического моделирования

In [31]:
for topic_id in range(min(10, len(topic_info))):
    size = topic_info[topic_info['Topic'] == topic_id]['Count'].values
    if len(size) > 0:
        print(f"Тема {topic_id}: {size[0]} документов")

Тема 0: 10897 документов
Тема 1: 4717 документов
Тема 2: 4239 документов
Тема 3: 3258 документов
Тема 4: 3020 документов
Тема 5: 1902 документов
Тема 6: 1514 документов
Тема 7: 1426 документов
Тема 8: 1248 документов
Тема 9: 1195 документов


In [14]:
cluster_model = HDBSCAN(min_cluster_size=500,
                        min_samples=50,
                        metric='euclidean',
                        cluster_selection_method='eom',
                        prediction_data=True
                        )

In [15]:
topic_model = BERTopic(
  embedding_model=embedding_model,
  umap_model=dim_model,
  hdbscan_model=cluster_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model,
  representation_model=representation_model,
  calculate_probabilities=True,
)

In [16]:
%%time

topics, probs = topic_model.fit_transform(train_docs)

CPU times: user 14min 19s, sys: 4.64 s, total: 14min 24s
Wall time: 10min 50s


In [17]:
topic_info = topic_model.get_topic_info()
num_topics = len(topic_info) - 1
print(f"Количество тем (без учета шума): {num_topics}")

Количество тем (без учета шума): 23


Часть задачи выполнено, удалось сократить количество тем более чем в 6 раз

In [18]:
topic_model.visualize_barchart(top_n_topics=40, n_words=10, title='Топ слов по темам')

In [21]:
topic_words = []
for topic_id in range(len(topic_model.get_topic_info()) - 1):
    words = topic_model.get_topic(topic_id)
    if words:
        top_words = [word for word, _ in words[:10]]
        topic_words.append(top_words)

diversity_score = topic_diversity(topic_words, top_n=10)
print(f"Topic Diversity: {diversity_score:.3f}")

Topic Diversity: 0.722


In [37]:
tokenized_texts = [doc.split() for doc in train_docs]

bertopic_topics = []
for topic_id in range(len(topic_model.get_topic_info()) - 1):
    words = topic_model.get_topic(topic_id)
    if words:
        bertopic_topics.append(words)


umass_score = umass_coherence_gensim(
    bertopic_topics,
    tokenized_texts,
    top_n=10
)
print(f"UMass Coherence: {umass_score:.3f}")

UMass Coherence: -4.256


Значание когерентночти также упало в 2 раза при небольшом относительно падении Diversity топиков. Дополнительно проверим покрытие сэмплов, не снизилось ли оно при изменении параметров кластеризации

In [24]:
before_coverage = pd.DataFrame([coverage_stats(topics)], index=["before"])
before_coverage

,total_docs,assigned_docs,outliers,coverage,outlier_rate
before,80000,54812,25188,0.68515,0.31485


In [25]:
reduced_topics_ctfidf = topic_model.reduce_outliers(
    train_docs,
    topics,
    strategy="c-tf-idf",
    threshold=0.1
)

after_coverage_ctfidf = pd.DataFrame([coverage_stats(reduced_topics_ctfidf)], index=["after_c_tf_idf"])
pd.concat([before_coverage, after_coverage_ctfidf])

,total_docs,assigned_docs,outliers,coverage,outlier_rate
before,80000,54812,25188,0.68515,0.31485
after_c_tf_idf,80000,59556,20444,0.74445,0.25555


По результатам использовании модели тематического моделирования можно констатировать, что используя достаточно простую настройку возможно действительно покрывать темами более 70% сэмплов данных. Достаточно сложно на данном датасете было получить четко оформленные калстеры, при визуальной оценке результатов топиков, сложно однозначно сказать чем многие друг от друга отличаются. В целом, предполагаю, что данную проблему можно было бы решить передавая top_k токенов по каждому кластеру в LLM для суммаризации и определение четких тем кластеров.